In [0]:

# ============================================================
# RetailPulse - Day 2: Customers Dataset Profiling
# ============================================================

# 1. Import PySpark functions
import pyspark.sql.functions as F


# 2. Define the dataset path
common_path = "/Volumes/retailpulse/dev/olist_raw"

customers_path = common_path + "/olist_customers_dataset.csv"


# 3. Load the customers CSV into a Spark DataFrame
customers_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(customers_path)
)


# 4. Display sample records
display(customers_df)


# 5. Inspect the schema
customers_df.printSchema()


# 6. Count total rows
customer_row_count = customers_df.count()

print("Total customer records:", customer_row_count)


# 7. Count NULL values in every column
customer_nulls = [
    F.coalesce(
        F.sum(
            F.when(F.col(c).isNull(), 1)
        ),
        F.lit(0)
    ).alias(c)
    for c in customers_df.columns
]

display(customers_df.select(customer_nulls))


# 8. Find duplicate customer_id groups
duplicate_customer_df = (
    customers_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_customer_count = duplicate_customer_df.count()

print("Duplicate customer_id groups:", duplicate_customer_count)


# 9. Find repeated customer_unique_id groups
duplicate_customerunq_df = (
    customers_df
    .groupBy("customer_unique_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_unique_id_count = duplicate_customerunq_df.count()

print("Repeated customer_unique_id groups:", duplicate_unique_id_count)


# 10. Count rows belonging to repeated customer_unique_id groups
repeated_customer_rows = (
    duplicate_customerunq_df
    .agg(F.sum("count").alias("total_rows"))
)

display(repeated_customer_rows)


# 11. Find the maximum occurrences of a customer_unique_id
max_unique_id_count = (
    duplicate_customerunq_df
    .agg(F.max("count").alias("max_occurrences"))
)

display(max_unique_id_count)


# 12. Check Distinct Records
custome_distinct_row_count=customers_df.distinct().count()
display("distinct_records_count : ", custome_distinct_row_count)

# 13. Duplicate Records Count
display("Duplicate Records Count : ",customer_row_count - custome_distinct_row_count)




# 1. Customers Dataset Profiling

## Objective
Analyze the Olist customers dataset to understand its structure, completeness, and uniqueness before processing it in the RetailPulse pipeline.

## Dataset Details
- **Source file:** `olist_customers_dataset.csv`
- **DataFrame:** `customers_df`
- **Total records:** 99,441

## Profiling Checks
1. Inspect sample records and schema.
2. Count total rows.
3. Calculate NULL counts for all columns.
4. Check duplicate `customer_id` groups.
5. Analyze repeated `customer_unique_id` values.
6. Check exact duplicate records.

## Key Findings
- Duplicate `customer_id` groups: 0
- Repeated `customer_unique_id` groups: 2,997
- Rows belonging to repeated-ID groups: 6,342
- Maximum occurrences of one `customer_unique_id`: 17
- Exact duplicate records: 0

## Data Quality Notes
Repeated `customer_unique_id` values are not automatically errors. They can represent multiple records associated with the same underlying customer. Review the NULL-count output before finalizing the data-quality assessment.

In [0]:

# ============================================================
# RetailPulse - Day 2: Orders Dataset Profiling
# ============================================================


# 1. Load the orders dataset
orders_path = common_path + "/olist_orders_dataset.csv"

orders_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(orders_path)
)


# 2. Display sample records
display(orders_df.limit(5))


# 3. Inspect schema
orders_df.printSchema()


# 4. Count total records
orders_row_count = orders_df.count()

print("Total Orders Records:", orders_row_count)


# 5. Count NULL values in every column
orders_nulls = [
    F.coalesce(
        F.sum(F.when(F.col(c).isNull(), 1)),
        F.lit(0)
    ).alias(c)
    for c in orders_df.columns
]

display(orders_df.select(orders_nulls))


# 6. Check duplicate order_id groups
duplicate_orderid_df = (
    orders_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_orderid_count = duplicate_orderid_df.count()

print("Duplicate order_id groups:", duplicate_orderid_count)


# 7. Check exact duplicate records
distinct_orders_count = orders_df.distinct().count()

duplicate_records_count = orders_row_count - distinct_orders_count

print("Distinct Orders Records:", distinct_orders_count)
print("Exact Duplicate Records:", duplicate_records_count)


# 8. Count orders by status
order_status_count_df = (
    orders_df
    .groupBy("order_status")
    .count()
    .orderBy(F.col("count").desc())
)

display(order_status_count_df)


# 9. Count missing customer-delivery timestamps by status
order_delivery_null_counts_df = (
    orders_df
    .groupBy("order_status")
    .agg(
        F.coalesce(
            F.sum(
                F.when(
                    F.col("order_delivered_customer_date").isNull(), 1
                )
            ),
            F.lit(0)
        ).alias("null_count")
    )
    .orderBy(F.col("null_count").desc())
)

display(order_delivery_null_counts_df)


# 10. Inspect delivered orders missing customer-delivery timestamps
delivered_missing_delivery_df = orders_df.filter(
    (F.col("order_status") == "delivered") &
    (F.col("order_delivered_customer_date").isNull())
)

display(delivered_missing_delivery_df)


# 11. Count missing approval timestamps by status
order_approval_nulls_by_status_df = (
    orders_df
    .groupBy("order_status")
    .agg(
        F.coalesce(
            F.sum(
                F.when(F.col("order_approved_at").isNull(), 1)
            ),
            F.lit(0)
        ).alias("null_count")
    )
    .orderBy(F.col("null_count").desc())
)

display(order_approval_nulls_by_status_df)


# 12. Inspect delivered orders missing approval timestamps
delivered_missing_approval_df = orders_df.filter(
    (F.col("order_status") == "delivered") &
    (F.col("order_approved_at").isNull())
)

display(delivered_missing_approval_df)


# 13. Count delivered orders later than their estimated delivery timestamp
late_delivered_orders_df = orders_df.filter(
    (F.col("order_status") == "delivered") &
    (
        F.col("order_delivered_customer_date") >
        F.col("order_estimated_delivery_date")
    )
)

late_delivered_orders_count = late_delivered_orders_df.count()

print("Orders delivered after estimated delivery timestamp:",
      late_delivered_orders_count)


# 2. Orders Dataset Profiling

## Objective
Profile the Olist orders dataset to assess data completeness, order identifier uniqueness, order status distribution, and delivery timestamp consistency.

## Dataset
- **Source file:** `olist_orders_dataset.csv`
- **Source location:** `/Volumes/retailpulse/dev/olist_raw/`
- **DataFrame:** `orders_df`

## Profiling Checks
1. Load the CSV and inspect sample records.
2. Examine column names and data types.
3. Calculate the total number of records.
4. Count NULL values in every column.
5. Check duplicate `order_id` groups.
6. Identify exact duplicate records across all columns.
7. Summarize orders by `order_status`.
8. Investigate missing customer-delivery timestamps by status.
9. Investigate missing approval timestamps by status.
10. Identify delivered orders with missing delivery or approval timestamps.
11. Count delivered orders whose actual delivery timestamp is later than their estimated delivery timestamp.

## Key Findings
- Total records: 99,441
- Duplicate `order_id` groups: 0
- Exact duplicate records: 0
- Missing `order_approved_at`: 160
- Missing `order_delivered_carrier_date`: 1,783
- Missing `order_delivered_customer_date`: 2,965
- Delivered orders missing customer-delivery timestamps: 8
- Delivered orders missing approval timestamps: 14
- Delivered orders later than their estimated delivery timestamps: 7,826

## Data Quality Notes
Missing timestamps must be interpreted in the context of order status. The eight delivered orders missing customer-delivery timestamps and the fourteen delivered orders missing approval timestamps require investigation. Do not automatically fill in missing timestamps or remove records.

The late-delivery metric compares full timestamps, not just calendar dates.

## Outcome
The orders dataset has been profiled for basic completeness, uniqueness, status distribution, and delivery-related inconsistencies. Findings will inform validation rules in the RetailPulse Silver layer.

In [0]:

# ============================================================
# 3. ORDER ITEMS DATASET PROFILING
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 3.1 Load Dataset and Inspect Schema
# ------------------------------------------------------------

order_items_path = common_path + "/olist_order_items_dataset.csv"

order_items_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(order_items_path)
)

order_items_df.printSchema()
order_items_df.limit(5).display()


# ------------------------------------------------------------
# 3.2 Total Record Count
# ------------------------------------------------------------

order_items_row_count = order_items_df.count()

print("Total Order Items Records:", order_items_row_count)


# ------------------------------------------------------------
# 3.3 NULL Value Check
# ------------------------------------------------------------

order_items_nulls = [
    F.coalesce(
        F.sum(F.when(F.col(c).isNull(), F.lit(1))),
        F.lit(0)
    ).alias(c)
    for c in order_items_df.columns
]

order_items_null_count_df = order_items_df.select(order_items_nulls)

order_items_null_count_df.display()


# ------------------------------------------------------------
# 3.4 Composite Key Uniqueness Check
# Composite key: (order_id, order_item_id)
# ------------------------------------------------------------

duplicate_order_item_keys_df = (
    order_items_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_order_item_keys_count = duplicate_order_item_keys_df.count()

print("Duplicate Composite Key Groups:", duplicate_order_item_keys_count)

# Display duplicate groups only if any exist
if duplicate_order_item_keys_count > 0:
    duplicate_order_item_keys_df.display()


# ------------------------------------------------------------
# 3.5 Exact Duplicate Row Check
# ------------------------------------------------------------

distinct_order_items_count = order_items_df.distinct().count()

exact_duplicate_order_items_count = (
    order_items_row_count - distinct_order_items_count
)

print("Distinct Full Rows:", distinct_order_items_count)
print("Exact Duplicate Rows:", exact_duplicate_order_items_count)


# ------------------------------------------------------------
# 3.6 Numeric Validation: Zero or Negative Values
# ------------------------------------------------------------

numerical_columns = ["price", "freight_value"]

order_items_non_positive_counts = [
    F.coalesce(
        F.sum(F.when(F.col(c) <= 0, F.lit(1))),
        F.lit(0)
    ).alias(c)
    for c in numerical_columns
]

order_items_non_positive_counts_df = order_items_df.select(
    order_items_non_positive_counts
)

order_items_non_positive_counts_df.display()


# ------------------------------------------------------------
# 3.7 Investigate Zero and Negative Freight Values
# ------------------------------------------------------------

zero_freight_count = (
    order_items_df
    .filter(F.col("freight_value") == 0)
    .count()
)

negative_freight_count = (
    order_items_df
    .filter(F.col("freight_value") < 0)
    .count()
)

print("Zero Freight Records:", zero_freight_count)
print("Negative Freight Records:", negative_freight_count)

# Inspect zero-freight records if needed
order_items_df.filter(
    F.col("freight_value") == 0
).select(
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value"
).limit(20).display()


# ------------------------------------------------------------
# 3.8 Numeric Summary Statistics
# Calculate min, max and average for each numeric column
# ------------------------------------------------------------

order_items_stats = []

for c in numerical_columns:
    order_items_stats.extend([
        F.min(F.col(c)).alias(f"min_{c}"),
        F.max(F.col(c)).alias(f"max_{c}"),
        F.avg(F.col(c)).alias(f"avg_{c}")
    ])

order_items_stats_df = order_items_df.select(order_items_stats)

order_items_stats_df.display()

# Convert the single-row summary into a Python dictionary
order_items_stats_dict = order_items_stats_df.first().asDict()


# ------------------------------------------------------------
# 3.9 Investigate Maximum Values Using Reusable Logic
# ------------------------------------------------------------

maximum_values = {
    "price": order_items_stats_dict["max_price"],
    "freight_value": order_items_stats_dict["max_freight_value"]
}

columns_to_inspect = [
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value"
]

for column_name, maximum_value in maximum_values.items():
    print(f"Records with maximum {column_name}: {maximum_value}")

    (
        order_items_df
        .filter(F.col(column_name) == maximum_value)
        .select(*columns_to_inspect)
        .display()
    )



# 3. Order Items Dataset Profiling

## Objective
Profile the Order Items dataset to understand its structure, validate data quality, verify composite key uniqueness, and analyze numerical columns before building the Bronze layer.

## Dataset Details
- **Source:** `olist_order_items_dataset.csv`
- **Total Records:** 112,650
- **Total Columns:** 7
- **Composite Key:** (`order_id`, `order_item_id`)

## Columns
- `order_id` — Unique identifier of an order
- `order_item_id` — Sequential item number within an order
- `product_id` — Identifier of the product
- `seller_id` — Identifier of the seller
- `shipping_limit_date` — Seller shipping deadline
- `price` — Price of the order item
- `freight_value` — Shipping cost associated with the order item

## Data Quality Checks

### 1. NULL Value Check
- No NULL values were found in any of the seven columns.
- All columns passed the NULL validation check.

### 2. Composite Key Validation
- Composite key: (`order_id`, `order_item_id`)
- Duplicate composite-key groups: 0
- The composite key passed the uniqueness check.

### 3. Exact Duplicate Row Check
- Distinct full rows: 112,650
- Exact duplicate rows: 0
- No exact duplicate rows were found.

### 4. Numeric Validation
- No zero or negative values were found in `price`.
- 383 records have `freight_value = 0`.
- No records have negative freight values.
- Zero freight values were retained as observations because they may represent valid free-shipping cases.

## Numerical Summary Statistics

| Metric | Price | Freight Value |
|---|---:|---:|
| Minimum | 0.85 | 0.00 |
| Maximum | 6,735.00 | 409.68 |
| Average | 120.65 | 19.99 |

Values are expressed in Brazilian reais (R$).

## Maximum-Value Investigation
- Maximum item price: R$6,735.00
- Maximum freight value: R$409.68
- Both maximum-value records were inspected.
- These values were documented for investigation and were not automatically classified as invalid.

## Conclusion
The Order Items dataset contains 112,650 records and passed the NULL, composite-key uniqueness, and exact duplicate-row checks.

The numerical checks identified 383 zero-freight records, with no negative freight values and no non-positive prices. The maximum price and freight values were inspected as part of exploratory profiling.

No records were deleted or modified during profiling.

## Next Steps
- Continue profiling the remaining source datasets.
- Build reusable profiling functions to reduce repeated code.
- Use the profiling findings to guide subsequent Bronze and Silver layer validation.



In [0]:

from pyspark.sql import functions as F

# ==================================================
# 1. LOAD DATASET
# ==================================================

orders_payment_path = common_path + "/olist_order_payments_dataset.csv"

orders_payment_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(orders_payment_path)
)

# Inspect sample records and schema
orders_payment_df.limit(5).display()
orders_payment_df.printSchema()


# ==================================================
# 2. BASIC PROFILING
# ==================================================

# Total number of records
total_records = orders_payment_df.count()
print("Total records:", total_records)

# NULL count for each column
all_columns_null = [
    F.coalesce(
        F.sum(F.when(F.col(c).isNull(), F.lit(1))),
        F.lit(0)
    ).alias(c)
    for c in orders_payment_df.columns
]

orders_payment_df.select(*all_columns_null).display()

# Exact duplicate rows
distinct_records = orders_payment_df.distinct().count()
exact_duplicate_rows = total_records - distinct_records

print("Distinct records:", distinct_records)
print("Exact duplicate rows:", exact_duplicate_rows)


# ==================================================
# 3. CANDIDATE KEY VALIDATION
# ==================================================

# Check whether order_id is unique
orders_payment_df.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .display()

# Check whether the composite key is unique
# Candidate key: (order_id, payment_sequential)
orders_payment_df.groupBy("order_id", "payment_sequential") \
    .count() \
    .filter(F.col("count") > 1) \
    .display()


# ==================================================
# 4. PAYMENT VALUE VALIDATION
# ==================================================

# Count zero payment values
zero_payment_count = orders_payment_df.filter(
    F.col("payment_value") == 0
).count()

print("Zero payment values:", zero_payment_count)

# Count negative payment values
negative_payment_count = orders_payment_df.filter(
    F.col("payment_value") < 0
).count()

print("Negative payment values:", negative_payment_count)

# Minimum, maximum, and average payment value
orders_payment_df.agg(
    F.min("payment_value").alias("min_payment_value"),
    F.max("payment_value").alias("max_payment_value"),
    F.avg("payment_value").alias("avg_payment_value")
).display()


# ==================================================
# 5. PAYMENT INSTALLMENT VALIDATION
# ==================================================

# Count zero installments
zero_installments_count = orders_payment_df.filter(
    F.col("payment_installments") == 0
).count()

print("Zero installment records:", zero_installments_count)

# Count negative installments
negative_installments_count = orders_payment_df.filter(
    F.col("payment_installments") < 0
).count()

print("Negative installment records:", negative_installments_count)


# ==================================================
# 6. PAYMENT ANALYSIS
# ==================================================

# 1. Payment type distribution
# Count records for each payment type

orders_payment_df.groupBy("payment_type") .count().orderBy(F.desc("count")) .display()


# 2. Payment value by type
# Calculate total payment value for each payment type

orders_payment_df.groupBy("payment_type") .agg(F.sum("payment_value").alias("total_payment_value") ).orderBy(F.desc("total_payment_value")).display()


# 3. Installment distribution
# Count records for each installment value

orders_payment_df.groupBy("payment_installments") .count() .orderBy("payment_installments") .display()


# RetailPulse --- Order Payments Dataset Profiling

## 1. Dataset Overview

-   **Source file:** `olist_order_payments_dataset.csv`
-   **DataFrame:** `orders_payment_df`
-   **Total records:** 103,886
-   **Columns:** 5
-   **Exact duplicate rows:** 0
-   **NULL values:** 0 across all five columns

## 2. Schema Observed

  Column                   Inferred data type
  ------------------------ --------------------
  `order_id`               String
  `payment_sequential`     Integer
  `payment_type`           String
  `payment_installments`   Integer
  `payment_value`          Double

> These types were observed using Spark schema inference. Final explicit
> schemas should be confirmed after profiling all nine source datasets.

## 3. NULL and Duplicate Checks

### NULL counts

  Column                     NULL count
  ------------------------ ------------
  `order_id`                          0
  `payment_sequential`                0
  `payment_type`                      0
  `payment_installments`              0
  `payment_value`                     0

### Duplicate checks

-   Exact duplicate rows: **0**
-   `order_id` is not unique because an order can have multiple payment
    records.
-   No duplicate groups were returned for the composite key
    **(`order_id`, `payment_sequential`)**.

The composite key is a strong candidate key for individual payment
records in this dataset.

## 4. Payment Value Validation

  Check                          Result
  ------------------------- -----------
  Zero payment values                 9
  Negative payment values             0
  Minimum payment value            0.00
  Maximum payment value       13,664.08
  Average payment value       154.10038

### Findings

-   Nine records have a zero payment value.
-   No negative payment values were found.
-   The maximum observed payment value is 13,664.08.
-   Zero-value records and the maximum-value record should be inspected
    before deciding whether any quality rule is needed.
-   Do not automatically delete or modify these records.

## 5. Installment Validation

  Check                            Result
  ------------------------------ --------
  Zero installment records              2
  Negative installment records          0

Two records have zero installments. No negative installment counts were
found. The two zero-installment records should be inspected before
deciding how to handle them.

## 6. Payment Type Distribution

  Payment type      Record count   Percentage (approx.)
  --------------- -------------- ----------------------
  `credit_card`           76,795                 73.92%
  `boleto`                19,784                 19.04%
  `voucher`                5,775                  5.56%
  `debit_card`             1,529                  1.47%
  `not_defined`                3                 0.003%
  **Total**          **103,886**               **100%**

Credit card is the most common payment type by record count. The three
`not_defined` records should be inspected rather than automatically
removed.

## 7. Total Payment Value by Type

  Payment type            Total payment value
  --------------------- ---------------------
  `credit_card`                 12,542,084.19
  `boleto`                       2,869,361.27
  `voucher`                        379,436.87
  `debit_card`                     217,989.79
  `not_defined`                          0.00
  **Total (approx.)**       **16,008,872.12**

Credit card has the largest total payment value. The dataset represents
Brazilian e-commerce, so the monetary values are understood to be in
Brazilian reais (R\$).

Floating-point aggregation may display small decimal artifacts; rounded
values are used in this report.

## 8. Installment Distribution

    Installments   Record count
  -------------- --------------
               0              2
               1         52,546
               2         12,413
               3         10,461
               4          7,098
               5          5,239
               6          3,920
               7          1,626
               8          4,268
               9            644
              10          5,328
              11             23
              12            133
              13             16
              14             15
              15             74
              16              5
              17              8
              18             27
              20             17
              21              3
              22              1
              23              1
              24             18

### Findings

-   One installment is the most common value, with 52,546 records.
-   The observed installment values range from 0 to 24.
-   Two records have zero installments and need further investigation.
-   Higher installment counts should be documented and assessed against
    business rules rather than assumed invalid.

## 9. Data Quality Follow-up

Investigate the following records during the cleaning and validation
stages:

1.  The 9 records with `payment_value == 0`.
2.  The 2 records with `payment_installments == 0`.
3.  The 3 records where `payment_type == 'not_defined'`.
4.  The record or records with the maximum payment value of 13,664.08.

Do not delete or overwrite these records without evidence and a
documented rule.

## 10. Profiling Status

-   [x] Load dataset and inspect schema
-   [x] Count records
-   [x] Check NULL counts
-   [x] Check exact duplicate rows
-   [x] Check candidate key uniqueness
-   [x] Validate payment values
-   [x] Validate installment counts
-   [x] Analyze payment types and total payment values
-   [x] Analyze installment distribution
-   [ ] Inspect flagged records in detail
-   [ ] Finalize explicit schema after profiling all nine datasets

## 11. Key Learning

-   `count()` counts records.
-   `distinct().count()` counts distinct full rows.
-   `total_records - distinct().count()` gives the number of extra exact
    duplicate rows.
-   `groupBy("order_id").count()` helps reveal repeated orders.
-   A composite key can identify a record when a single column is not
    unique.
-   `sum("payment_value")` calculates total value per payment type.
-   Unusual values should be investigated and documented before
    cleaning.


In [0]:

from pyspark.sql import functions as F

# ============================================================
# 1. REUSABLE DATASET PROFILING FUNCTION
# ============================================================

def profile_dataset(dataframe):

    # Basic dataset information
    total_records = dataframe.count()
    total_columns = len(dataframe.columns)

    # Collect column names and data types
    column_data_types = {}

    for field in dataframe.schema.fields:
        column_data_types[field.name] = field.dataType

    # Calculate NULL counts for all columns in one aggregation
    null_count_expressions = [
        F.coalesce(
            F.sum(
                F.when(F.col(column_name).isNull(), F.lit(1))
            ),
            F.lit(0)
        ).alias(column_name)
        for column_name in column_data_types.keys()
    ]

    null_counts = dataframe.agg(
        *null_count_expressions
    ).first().asDict()

    # Calculate exact duplicate rows
    distinct_row_count = dataframe.distinct().count()
    duplicate_row_count = total_records - distinct_row_count

    # Build column-level profiles
    column_profiles = {}

    for column_name, data_type in column_data_types.items():
        column_profiles[column_name] = {
            "data_type": data_type,
            "null_count": null_counts[column_name]
        }

    # Return structured profiling results
    return {
        "total_records": total_records,
        "total_columns": total_columns,
        "distinct_rows": distinct_row_count,
        "duplicate_rows": duplicate_row_count,
        "column_profile": column_profiles
    }


# ============================================================
# 2. DEFINE SOURCE DATASET PATHS
# ============================================================

common_path = "/Volumes/retailpulse/dev/olist_raw"

datasets = {
    "customers": common_path + "/olist_customers_dataset.csv",
    "orders": common_path + "/olist_orders_dataset.csv",
    "order_items": common_path + "/olist_order_items_dataset.csv",
    "order_payments": common_path + "/olist_order_payments_dataset.csv",
    "order_reviews": common_path + "/olist_order_reviews_dataset.csv",
    "products": common_path + "/olist_products_dataset.csv",
    "sellers": common_path + "/olist_sellers_dataset.csv",
    "geolocation": common_path + "/olist_geolocation_dataset.csv",
    "product_category_name_translation":
        common_path + "/product_category_name_translation.csv"
}


# ============================================================
# 3. LOAD ALL CSV FILES INTO SPARK DATAFRAMES
# ============================================================

dataframes = {}

for dataset_name, dataset_path in datasets.items():

    dataframes[dataset_name] = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(dataset_path)
    )

print("Total datasets loaded:", len(dataframes))


# ============================================================
# 4. PROFILE ALL DATASETS
# ============================================================

all_profiles = {}

for dataset_name, dataframe in dataframes.items():

    print(f"Profiling dataset: {dataset_name}")

    profile = profile_dataset(dataframe)

    all_profiles[dataset_name] = profile

print("Total datasets profiled:", len(all_profiles))


# ============================================================
# 5. BUILD CONSOLIDATED DATA QUALITY SUMMARY
# ============================================================

summary_rows = []

for dataset_name, profile in all_profiles.items():

    # Calculate total NULL values across all columns
    total_nulls = sum(
        column_details["null_count"]
        for column_details
        in profile["column_profile"].values()
    )

    # Build one summary record per dataset
    summary_rows.append({
        "dataset_name": dataset_name,
        "total_records": profile["total_records"],
        "total_columns": profile["total_columns"],
        "total_nulls": total_nulls,
        "duplicate_rows": profile["duplicate_rows"]
    })


# ============================================================
# 6. CREATE AND DISPLAY SUMMARY TABLE
# ============================================================

summary_df = spark.createDataFrame(summary_rows)

display(
    summary_df.orderBy("dataset_name")
)

summary_df = summary_df.withColumn(
    "null_percentage",
    F.round(
        (F.col("total_nulls") /
         (F.col("total_records") * F.col("total_columns"))) * 100,
        2
    )
)

display(summary_df.orderBy("dataset_name"))
